[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vvvvvivekkk/AI-Engineer/blob/main/11-ai-automation/notebooks/01-automation-vs-agents-python.ipynb)

# Notebook 01 — Automation vs agents, and Python automation

**Goal:** feel the difference between a fixed **workflow** and an **agent** on the same job, then build a small, robust Python automation (fetch → filter → summarise → save) with **retries**, **timeouts** and a **schedule**.

Everything runs offline: network calls go to a *mock* transport (a pretend internet inside Python), and the "LLM" is a stand-in function that counts how often it is called.

## Part 1 — Same job, two designs

**Job:** route 6 support tickets to `billing`, `tech` or `general`.

We count **LLM calls**, because each real call costs money and time. First, the stand-in LLM: it "understands" a ticket using keywords, and counts its calls.

In [1]:
import time

LLM_CALLS = 0
def fake_llm(prompt: str) -> str:
    """Stand-in for a chat model. Returns a category word for classification prompts."""
    global LLM_CALLS
    LLM_CALLS += 1
    time.sleep(0.02)                         # pretend network latency
    p = prompt.lower()
    if "refund" in p or "charged" in p or "invoice" in p: return "billing"
    if "error" in p or "crash" in p or "login" in p:      return "tech"
    return "general"

tickets = ["I was charged twice", "App crashes on start", "Do you ship to Taiwan?",
           "Invoice missing VAT number", "Login error 403", "Love the product!"]
print(len(tickets), "tickets")

6 tickets


**Design A — workflow with one AI step.** The steps are fixed: classify (1 LLM call) → route with a normal `if`. The LLM does only the fuzzy part.

In [2]:
LLM_CALLS = 0
t0 = time.time()
routes = {}
for t in tickets:
    category = fake_llm(f"Classify this ticket as billing/tech/general: {t}")   # the only AI step
    routes.setdefault(category, []).append(t)                                    # plain code decides the rest
print(routes)
print(f"Workflow: {LLM_CALLS} LLM calls, {time.time()-t0:.2f}s")

{'billing': ['I was charged twice', 'Invoice missing VAT number'], 'tech': ['App crashes on start', 'Login error 403'], 'general': ['Do you ship to Taiwan?', 'Love the product!']}
Workflow: 6 LLM calls, 0.12s


**Design B — an agent.** The LLM is in control of every step: it decides to read the ticket, think, look up the team, then route. Realistic agent loops take several calls per ticket (here: 4).

In [3]:
LLM_CALLS = 0
t0 = time.time()
def agent_route(ticket):
    fake_llm(f"Plan: how should I handle '{ticket}'?")          # step 1: plan
    fake_llm(f"Which tool do I need for '{ticket}'?")           # step 2: choose tool
    category = fake_llm(f"Classify: {ticket}")                   # step 3: classify
    fake_llm(f"Write the final routing decision for {category}") # step 4: finish
    return category
agent_routes = {}
for t in tickets:
    agent_routes.setdefault(agent_route(t), []).append(t)
print(agent_routes == routes, "-> same result")
print(f"Agent: {LLM_CALLS} LLM calls, {time.time()-t0:.2f}s")

True -> same result
Agent: 24 LLM calls, 0.49s


Same answer, **4× the calls**. For a well-understood job, the workflow wins on cost, speed and predictability. Agents earn their cost when the steps *can't* be written in advance.

## Part 2 — A robust fetch with retries and timeouts

Real networks fail. We use `httpx` with a **MockTransport**: a fake internet that we control. It fails twice (HTTP 503 "service unavailable") and then succeeds, so we can watch the retry logic work.

In [4]:
import httpx

SAMPLE_RSS = """<rss><channel>
<item><title>New open-source LLM agent framework released</title><link>https://example.com/1</link></item>
<item><title>Gardening tips for rainy winters</title><link>https://example.com/2</link></item>
<item><title>Python 3.14 makes AI tooling faster</title><link>https://example.com/3</link></item>
<item><title>Local bakery wins award</title><link>https://example.com/4</link></item>
</channel></rss>"""

attempts_seen = {"n": 0}
def flaky_server(request: httpx.Request) -> httpx.Response:
    attempts_seen["n"] += 1
    if attempts_seen["n"] <= 2:
        return httpx.Response(503, text="busy")
    return httpx.Response(200, text=SAMPLE_RSS)

client = httpx.Client(transport=httpx.MockTransport(flaky_server), timeout=10)
print("transport ready")

transport ready


**Retry with exponential backoff:** wait 0.1s, 0.2s, 0.4s… between attempts (real code would use seconds). `raise_for_status()` turns 4xx/5xx replies into exceptions we can catch.

In [5]:
def fetch_with_retry(url, attempts=4, backoff=0.1):
    for i in range(attempts):
        try:
            r = client.get(url)
            r.raise_for_status()
            print(f"attempt {i+1}: OK")
            return r.text
        except httpx.HTTPError as e:
            wait = backoff * 2 ** i
            print(f"attempt {i+1}: failed ({e.__class__.__name__}), waiting {wait:.1f}s")
            time.sleep(wait)
    raise RuntimeError("gave up after retries")

xml_text = fetch_with_retry("https://example.com/feed.xml")
print(len(xml_text), "characters received")

attempt 1: failed (HTTPStatusError), waiting 0.1s
attempt 2: failed (HTTPStatusError), waiting 0.2s


attempt 3: OK
417 characters received


## Part 3 — Parse, filter, summarise, save

**RSS** is XML. Python's built-in `xml.etree.ElementTree` reads it. We keep only items with a keyword as a **whole word** (so `ai` does not match `rainy`).

In [6]:
import xml.etree.ElementTree as ET

items = [{"title": it.findtext("title"), "link": it.findtext("link")}
         for it in ET.fromstring(xml_text).iter("item")]
KEYWORDS = {"ai", "llm", "agent", "python"}

def relevant(title):
    words = {w.strip(".,:;!?").lower() for w in title.split()}
    return bool(words & KEYWORDS)

kept = [it for it in items if relevant(it["title"])]
for it in kept: print("KEEP", it["title"])
print(len(items) - len(kept), "items dropped")

KEEP New open-source LLM agent framework released
KEEP Python 3.14 makes AI tooling faster
2 items dropped


The summary step is where an LLM would go (guarded by an API-key check, as always). Then we save to CSV with pandas — the Python twin of n8n's "Append to Google Sheet" node.

In [7]:
import os, pandas as pd
from datetime import date

def summarise(title):
    if os.getenv("ANTHROPIC_API_KEY"):
        import anthropic
        m = anthropic.Anthropic().messages.create(model=os.getenv("ANTHROPIC_MODEL", "claude-opus-5-5"), max_tokens=60,
            messages=[{"role": "user", "content": f"Summarise in 12 words: {title}"}])
        return m.content[0].text
    return "Summary: " + title[:50]          # offline stand-in

rows = [{"date": str(date.today()), "title": it["title"], "summary": summarise(it["title"]), "link": it["link"]} for it in kept]
df = pd.DataFrame(rows)
import tempfile, pathlib
out = pathlib.Path(tempfile.gettempdir()) / "digest_demo.csv"   # temp folder keeps the repo clean
df.to_csv(out, index=False)
print("saved to", out)
df

saved to /tmp/digest_demo.csv


,date,title,summary,link
0,2026-11-21,New open-source LLM agent framework released,Summary: New open-source LLM agent framework r...,https://example.com/1
1,2026-11-21,Python 3.14 makes AI tooling faster,Summary: Python 3.14 makes AI tooling faster,https://example.com/3


## Part 4 — Scheduling

The `schedule` library registers jobs in plain English. In a real script you loop forever calling `schedule.run_pending()`. Here we register jobs, look at when they would next run, and use `run_all()` to fire them once immediately (so the notebook finishes).

In [8]:
try:
    import schedule
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "schedule"], check=True)
    import schedule

schedule.clear()
def digest_job():
    print("digest job ran at", time.strftime("%H:%M:%S"))
schedule.every().day.at("08:00").do(digest_job)
schedule.every(15).minutes.do(lambda: print("health check ran"))
for j in schedule.get_jobs():
    print(j, "| next run:", j.next_run)
schedule.run_all()       # run every job now, once
schedule.clear()

Job(interval=1, unit=days, do=digest_job, args=(), kwargs={}) | next run: 2026-11-22 08:00:00
Job(interval=15, unit=minutes, do=<lambda>, args=(), kwargs={}) | next run: 2026-11-21 23:07:52.326917
digest job ran at 22:52:52
health check ran


`schedule` only works while your Python program is running. For jobs that must survive restarts, use **cron** (Linux/macOS), **Task Scheduler** (Windows), a cloud scheduler, or n8n's Schedule Trigger. A cron line has five time fields — minute, hour, day of month, month, day of week:

```
# m  h  dom mon dow  command
  0  8  *   *   *    /usr/bin/python3 /home/me/python_automation.py --once
```

This means "at minute 0 of hour 8, every day". Let's decode a few cron lines with a tiny helper.

In [9]:
def explain_cron(line):
    m, h, dom, mon, dow = line.split()[:5]
    days = {"*": "every day", "1-5": "Monday to Friday", "1": "Mondays", "0": "Sundays"}
    when = f"at {h.zfill(2)}:{m.zfill(2)}" if m.isdigit() and h.isdigit() else f"minute={m}, hour={h}"
    return f"{when}, {days.get(dow, 'dow=' + dow)}" + ("" if dom == "*" else f", day {dom} of the month")

for line in ["0 8 * * *", "30 18 * * 1-5", "0 9 1 * *"]:
    print(f"{line:15} -> {explain_cron(line)}")

0 8 * * *       -> at 08:00, every day
30 18 * * 1-5   -> at 18:30, Monday to Friday
0 9 1 * *       -> at 09:00, every day, day 1 of the month


## Recap

* Workflow + one AI step usually beats an agent for repeatable jobs (fewer calls, same result).
* Always use **timeouts**, **retries with backoff**, and clear failure handling.
* `schedule` for in-process timing; **cron**/cloud schedulers/n8n for jobs that must always run.

**Try it:** make `flaky_server` fail 5 times and see what happens with `attempts=4`. Then add a fallback that returns an empty feed and logs a warning instead of crashing.